# Notebook 01 — Data ingestion & quality audit

## Business question

Did a **simulated regional surge/incentive rollout** (high-volume Brazilian states after a policy cutoff) improve **on-time delivery**?

We cannot answer that in this notebook yet. First we need trustworthy **order-level data**: one row per order, valid delivery timestamps, and join keys that won't inflate row counts when we build the analytical mart.

## What this notebook does

| Phase | Goal | Status |
|-------|------|--------|
| 1 | Load raw Olist CSVs and audit table grain / keys | In progress |
| 2 | Load tables into DuckDB (`data/olist.db`) | Not started |
| 3 | Build `orders_analytical` (one row per `order_id`) | Not started |
| 4 | Outcome EDA (supports DiD design) | Not started |

**Primary outcomes (defined in the mart, not here yet):**
- `on_time` — delivered on or before estimated delivery date
- `delivery_days` — days from purchase to customer delivery
- `customer_state` — region for treated vs control groups

> Full schema and join rules: [`docs/DATA_MODEL.md`](../docs/DATA_MODEL.md)


## Notebook structure

1. **Setup & load** — imports, paths, read 9 CSVs
2. **`orders` audit** — dtypes, null logic, primary key, date range *(spine of the mart)*
3. **`order_items` audit** — fan-out check *(must aggregate before join)*
4. **`customers` audit** — join key to orders, state for regional analysis
5. **Next** — DuckDB load → `orders_analytical` → EDA

### Target mart grain

`orders_analytical`: **one row per `order_id`**

If we join `order_items` or `payments` without aggregating first, one order becomes many rows and every downstream metric (on-time rate, DiD) is wrong.

## 1. Setup & data load

Load all 9 Olist CSVs into pandas for inspection. We persist to DuckDB in a later section; this phase is about understanding grain and data quality before any joins.

In [2]:
import duckdb, pandas as pd 
from pathlib import Path
print("OK")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

OK


In [3]:

customers =pd.read_csv(ROOT / "data/raw/olist/olist_customers_dataset.csv")
orders = pd.read_csv(ROOT / "data/raw/olist/olist_orders_dataset.csv")
order_items = pd.read_csv(ROOT / "data/raw/olist/olist_order_items_dataset.csv")
payments = pd.read_csv(ROOT / "data/raw/olist/olist_order_payments_dataset.csv")
reviews = pd.read_csv(ROOT / "data/raw/olist/olist_order_reviews_dataset.csv")
products = pd.read_csv(ROOT / "data/raw/olist/olist_products_dataset.csv")
product_category_name_translation = pd.read_csv(ROOT / "data/raw/olist/product_category_name_translation.csv")
geolocation = pd.read_csv(ROOT / "data/raw/olist/olist_geolocation_dataset.csv")
sellers = pd.read_csv(ROOT / "data/raw/olist/olist_sellers_dataset.csv")








## 2. `orders` table audit

`orders` is the **spine** of our mart: one row per order. We check four things before building joins:

1. **Types** — timestamps must be datetime to compute `delivery_days`
2. **Nulls** — missing delivery dates should align with non-delivered status
3. **Primary key** — `order_id` must be unique (no fan-out from the spine itself)
4. **Date range** — confirms we have enough pre/post history for DiD

In [4]:

orders.head()


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [5]:
print(orders.dtypes)


order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object


### 2a. Parse timestamps

**Why:** CSVs load date columns as strings. We need datetimes to compute `delivery_days` and compare delivery vs estimated dates.

In [6]:
time_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in time_cols: 
    orders[col] = pd.to_datetime(orders[col])

orders.dtypes




order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [7]:
print(orders.isnull().sum())
print(orders.shape)

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
(99441, 8)


### 2b. Null audit — delivery timestamp vs status

**Why:** `order_delivered_customer_date` drives our primary outcome (`on_time`). We need to know whether nulls mean "not delivered yet" (expected) or data errors (bad).

**Check:** Crosstab `order_status` × missing delivery timestamp.

In [8]:
orders['order_status'].unique()

<StringArray>
[  'delivered',    'invoiced',     'shipped',  'processing', 'unavailable',
    'canceled',     'created',    'approved']
Length: 8, dtype: str

*(Reference)* Distinct `order_status` values in the crosstab below:

In [9]:
# Null audit: order_delivered_customer_date vs order_status
missing_delivered_ts = orders["order_delivered_customer_date"].isna()

print("Crosstab (columns: False=has timestamp, True=missing)")
print(
    pd.crosstab(orders["order_status"], missing_delivered_ts, margins=True).to_string()
)


Crosstab (columns: False=has timestamp, True=missing)
order_delivered_customer_date  False  True    All
order_status                                     
approved                           0     2      2
canceled                           6   619    625
created                            0     5      5
delivered                      96470     8  96478
invoiced                           0   314    314
processing                         0   301    301
shipped                            0  1107   1107
unavailable                        0   609    609
All                            96476  2965  99441


#### So what — delivery timestamp nulls

- **2,965** null `order_delivered_customer_date`; almost all are **non-delivered** statuses (see crosstab).
- **8** delivered orders still have null timestamp (**~0.01%**) — treat as exceptions.

**Mart rule:** compute `delivery_days` / `on_time` only when `order_status == 'delivered'` and delivery timestamp is not null; otherwise NULL.

### 2c. Primary key check

**Why:** The mart must be one row per `order_id`. Duplicate `order_id` in `orders` would break every downstream metric.

In [10]:
print(orders['order_id'].duplicated().sum()) # expect 0 
print(orders['order_id'].nunique() == orders.shape[0]) # expect True 

0
True


### 2d. Purchase date range

**Why:** DiD needs a believable pre period and post period around a policy cutoff. This bounds what cutoffs are feasible.

In [11]:
orders['order_purchase_timestamp'].min(), orders['order_purchase_timestamp'].max()

(Timestamp('2016-09-04 21:15:19'), Timestamp('2018-10-17 17:30:18'))

#### So what — `orders` summary

| Check | Result | Implication |
|-------|--------|-------------|
| Grain | 99,441 rows; `order_id` unique | Safe spine for mart |
| Null delivery ts | Structural for non-delivered; 8 exceptions | Rule for outcomes documented above |
| Purchase window | 2016-09-04 → 2018-10-17 | ~2 years for pre/post DiD |

## 3. `order_items` table audit

**Grain:** one row per line item (`order_id` + `order_item_id`).

**Why we care:** Multiple items per order → **must aggregate to order level before joining** to `orders`, or we duplicate orders and corrupt on-time rates.

In [16]:
order_items.head(4)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79


In [27]:
print("Shape:", order_items.shape)
print("Items per order:")
print(order_items.groupby("order_id").size().describe())
print("Duplicate (order_id, order_item_id):", order_items.duplicated(subset=["order_id", "order_item_id"]).sum())

Rows and columns in order: (112650, 7)
count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
dtype: float64
0


#### So what — `order_items`

| Check | Result | Implication |
|-------|--------|-------------|
| Grain | PK = (`order_id`, `order_item_id`); 0 dupes | Line-item level |
| Items per order | median **1**, max **21** | Aggregate before join (`n_items`, `gross_revenue`, `freight_total`) |
| Rows | 112,650 vs 99,441 orders | Joining raw → fan-out |

## 4. `customers` table audit

**Grain:** one row per `customer_id` (Olist assigns a new `customer_id` per order).

**Why we care:** Join on `orders.customer_id` to get **`customer_state`** — the regional dimension for treated vs control groups in DiD.

In [28]:
customers.head(5)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [34]:
print("Shape:", customers.shape)
print("Duplicate customer_id:", customers.duplicated("customer_id").sum())
print(customers.groupby("customer_id").size().describe())

Rows and columns in order: (99441, 5)
0
count    99441.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
dtype: float64


#### So what — `customers`

| Check | Result | Implication |
|-------|--------|-------------|
| PK | `customer_id` unique; 0 dupes | 1:1 join to `orders` |
| Rows | 99,441 (= orders count) | Every order has exactly one customer record |
| Region | `customer_state` available | Use for simulated policy regions (e.g. SP, RJ, MG) |

---

## 5. Phase 1 findings & next steps

### What we validated (so far)

- **`orders`** is a clean spine: unique `order_id`, ~2 years of purchases, structural nulls on delivery timestamps.
- **`order_items`** and **`payments`** (see [`DATA_MODEL.md`](../docs/DATA_MODEL.md)) are **below order grain** — aggregate before joining.
- **`customers`** joins 1:1 and supplies **`customer_state`** for regional analysis.

### Next (same notebook, upcoming sections)

1. **DuckDB load** — persist 5 mart-critical tables to `data/olist.db`
2. **`orders_analytical`** — SQL mart at order grain with outcomes (`on_time`, `delivery_days`)
3. **Outcome EDA** — time series and distributions that answer: *can DiD work here?*

## 6. Load into DuckDB

In [35]:
DB_PATH = ROOT / "data/olist.db"
con = duckdb.connect(DB_PATH)

con.execute("CREATE OR REPLACE TABLE orders AS SELECT * FROM orders")
con.execute("CREATE OR REPLACE TABLE customers AS SELECT * FROM customers")
con.execute("CREATE OR REPLACE TABLE order_items AS SELECT * FROM order_items")
con.execute("CREATE OR REPLACE TABLE order_payments AS SELECT * FROM payments")
con.execute("CREATE OR REPLACE TABLE order_reviews AS SELECT * FROM reviews")

con.sql("SHOW TABLES").df()

,name
0,customers
1,order_items
2,order_payments
3,order_reviews
4,orders


In [40]:
con.sql("""
  SELECT 'orders' AS tbl, COUNT(*) AS n FROM orders
  UNION ALL SELECT 'customers', COUNT(*) FROM customers
  UNION ALL SELECT 'order_items', COUNT(*) FROM order_items
  UNION ALL SELECT 'order_payments', COUNT(*) FROM order_payments
  UNION ALL SELECT 'order_reviews', COUNT(*) FROM order_reviews
""").df()

,tbl,n
0,orders,99441
1,customers,99441
2,order_items,112650
3,order_payments,103886
4,order_reviews,99224


In [42]:
orders.head(5)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26
